# State Discrimination

The QiController allows flexible state discrimination from down-converted and integrated IQ blobs.

This is facilitates using `N` linear discriminators and a user-configurable Finite State Machine.

This notebooks covers simple cases and more complex ones

In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, LogNorm, Normalize, to_rgb
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1 import make_axes_locatable


@dataclass(frozen=True)
class BlobConfig:
    center: complex
    """
    Center of the blob. Radius must be <= 1, angle can be arbitrary
    """
    chance: float
    """
    Chance from 0-1 that this blob appears.
    Sum of all blobs must be 1
    """
    sigma: float | tuple[float, float]
    """
    Standard deviation.
    A single value describes a circular blob, a pair of values describes an elliptical
    one with the standard deviations along its two principal axes.
    """
    theta: float = 0.0
    """
    Angle in radians between the first principal axis of the blob and the I axis.
    Only meaningful for elliptical blobs.
    """

    def __post_init__(self):
        if not 0 <= self.chance <= 1:
            raise ValueError(
                f"Blob chance must be between 0 and 1 but it is {self.chance}"
            )
        if abs(self.center) > 1:
            raise ValueError(
                f"Blob center must lie within the unit circle but its radius is {abs(self.center)}"
            )
        if any(sigma < 0 for sigma in self.sigmas):
            raise ValueError(
                f"Blob standard deviations must not be negative but they are {self.sigmas}"
            )

    @property
    def sigmas(self) -> tuple[float, float]:
        """
        The standard deviations along both principal axes, also for circular blobs.
        """
        if isinstance(self.sigma, (int, float)):
            return float(self.sigma), float(self.sigma)
        if isinstance(self.sigma, tuple) and len(self.sigma) == 2:
            return float(self.sigma[0]), float(self.sigma[1])
        raise RuntimeError("sigma must be float or tuple of two floats")


def synthetic_iq_blobs(
    *configs: BlobConfig, count: int = 100000, seed=None, return_labels: bool = False
):
    """
    Generates `count` synthetic IQ samples drawn from the mixture of gaussian blobs
    described by `configs`.

    Every sample is assigned to one of the blobs with the probability given by its
    `chance` and is then drawn from a 2D gaussian around that blob's `center` with the
    standard deviations given by its `sigma`, rotated by its `theta`::

        data, labels = synthetic_iq_blobs(
            BlobConfig(center=0.5 + 0.5j, chance=0.7, sigma=0.05),
            BlobConfig(
                center=-0.4 - 0.2j, chance=0.3, sigma=(0.1, 0.02), theta=np.pi / 4
            ),
            return_labels=True,
        )

    :param configs: the blobs to draw from. Their chances must sum up to 1.
    :param count: the number of samples to generate.
    :param seed: seed of the random number generator. Pass a fixed value to obtain
        reproducible data.
    :param return_labels: whether to also return which blob every sample came from.
        Use this to compare a discriminator against the ground truth, e.g.
        `(discriminator.get_state(data) != labels).mean()` is the assignment error.

    :return:
        a complex array of shape `(count,)` containing I in the real and Q in the
        imaginary part. If `return_labels` is set, a tuple of that array and an
        integer array of shape `(count,)` indexing into `configs`.
    """
    if not configs:
        raise ValueError("At least one blob configuration is required")
    chances = np.array([config.chance for config in configs], dtype=float)
    if not np.isclose(chances.sum(), 1):
        raise ValueError(
            f"Blob chances must sum up to 1 but they sum up to {chances.sum()}"
        )
    centers = np.array([config.center for config in configs], dtype=complex)
    sigmas = np.array([config.sigmas for config in configs], dtype=float)
    thetas = np.array([config.theta for config in configs], dtype=float)

    rng = np.random.default_rng(seed)
    # Pick the blob every sample originates from, then scatter it around that center
    # along the blob's principal axes and rotate the result onto the IQ plane.
    labels = rng.choice(len(configs), size=count, p=chances / chances.sum())
    noise = rng.normal(scale=sigmas[labels, 0]) + 1j * rng.normal(
        scale=sigmas[labels, 1]
    )
    data = centers[labels] + noise * np.exp(1j * thetas[labels])
    if return_labels:
        return data, labels
    return data


def _square_limits(data, limits):
    """
    Returns the I and Q range of a square window around `data`, or `limits` for both
    axes if that is given.
    """
    if limits is not None:
        return limits, limits
    center_i = (data.real.max() + data.real.min()) / 2
    center_q = (data.imag.max() + data.imag.min()) / 2
    radius = max(np.ptp(data.real), np.ptp(data.imag)) / 2 * 1.1
    # All samples may lie in a single point, e.g. one single blob with sigma == 0.
    radius = radius or 1.0
    return (
        (center_i - radius, center_i + radius),
        (center_q - radius, center_q + radius),
    )


def _blob_colormap(color):
    """
    Returns a colormap that fades from transparent to `color`, so that several blobs
    can be drawn over each other.
    """
    rgb = to_rgb(color)
    # Start at a visible alpha, otherwise the sparsest bins disappear entirely.
    return LinearSegmentedColormap.from_list("blob", [(*rgb, 0.25), (*rgb, 1.0)])


def plot_iq_blobs(
    data,
    labels=None,
    ax=None,
    size: float = 6.0,
    limits: tuple[float, float] | None = None,
    bins: int = 100,
    log: bool = True,
    **kwargs,
):
    """
    Plots the density of IQ data on a square plot of the IQ plane.

    The samples are binned into a 2D histogram whose counts are colored on a
    logarithmic scale by default, so that sparsely populated blobs and the tails of
    dense ones stay visible next to a blob that holds most of the samples.

    Both axes span the same range so that circular blobs also look circular. A
    `LinearDiscriminator` can be drawn on top of the returned axis::

        data, labels = synthetic_iq_blobs(state_0, state_1, return_labels=True)
        ax = plot_iq_blobs(data, labels)
        LinearDiscriminator.estimate(data[labels == 0], data[labels == 1]).plot(ax)

    :param data: complex IQ samples, e.g. as returned by `synthetic_iq_blobs`.
    :param labels: the blob every sample came from. If given, the blobs are drawn in
        separate colors with a legend instead of in one colormap with a colorbar.
    :param ax: the `matplotlib.axes.Axes` to draw on. A new square figure is created
        if this is `None`.
    :param size: edge length of the newly created figure in inches. Unused if `ax` is
        given.
    :param limits: the range of both the I and the Q axis. Defaults to a square window
        around the data.
    :param bins: the number of histogram bins along each axis. Fewer bins mean more
        samples per bin and therefore a larger range of the color scale.
    :param log: whether to scale the color logarithmically. Set this to `False` to
        color the counts linearly, which shows the shape of the largest blob but
        drowns out blobs with a low chance.
    :param kwargs: further keyword arguments passed on to
        `matplotlib.axes.Axes.pcolormesh`.

    :return:
        the `matplotlib.axes.Axes` that was drawn on.
    """
    data = np.asarray(data)
    if ax is None:
        _, ax = plt.subplots(figsize=(size, size))

    limits_i, limits_q = _square_limits(data, limits)
    edges_i = np.linspace(*limits_i, bins + 1)
    edges_q = np.linspace(*limits_q, bins + 1)

    def histogram(samples):
        counts, _, _ = np.histogram2d(
            samples.real, samples.imag, bins=(edges_i, edges_q)
        )
        # Empty bins are masked so that they stay transparent instead of being drawn
        # as the lowest color, which also keeps them out of the logarithmic scale.
        return np.ma.masked_equal(counts.T, 0)

    if labels is None:
        histograms = {None: histogram(data)}
    else:
        labels = np.asarray(labels)
        histograms = {
            label: histogram(data[labels == label]) for label in np.unique(labels)
        }

    # A shared scale over all blobs keeps their densities comparable.
    vmax = max(counts.max() for counts in histograms.values())
    if log:
        # vmax must stay above vmin even if no bin holds more than one sample.
        norm = LogNorm(vmin=1, vmax=max(vmax, 2))
    else:
        norm = Normalize(vmin=0, vmax=max(vmax, 1))

    for index, (label, counts) in enumerate(histograms.items()):
        if label is None:
            cmap = plt.get_cmap("viridis").copy()
        else:
            cmap = _blob_colormap(f"C{index}")
        cmap.set_bad(alpha=0)
        mesh = ax.pcolormesh(edges_i, edges_q, counts, norm=norm, cmap=cmap, **kwargs)

    if labels is None:
        # The axis is quadratic and therefore lower than its slot in the figure, so
        # the colorbar is attached to the axis itself to end up exactly as high.
        cax = make_axes_locatable(ax).append_axes("right", size="4%", pad=0.15)
        ax.figure.colorbar(mesh, cax=cax, label="samples per bin")
    else:
        ax.legend(
            handles=[
                Patch(color=f"C{index}", label=f"blob {label}")
                for index, label in enumerate(histograms)
            ],
            loc="upper right",
        )

    ax.set_xlim(limits_i)
    ax.set_ylim(limits_q)
    ax.set_aspect("equal")

    ax.set_xlabel("I")
    ax.set_ylabel("Q")
    ax.grid(alpha=0.2)
    return ax

## Example 1: Simple two-state discrimination

In [ ]:
from qiclib.hardware.recording import StateConfig
from qiclib.state_estimation import LinearDiscriminator

state0 = BlobConfig(center=0.5 + 0.1j, chance=0.8, sigma=0.1)
state1 = BlobConfig(center=-0.3 + 0.1j, chance=0.2, sigma=0.1)

iq, labels = synthetic_iq_blobs(state0, state1, return_labels=True)

ld = LinearDiscriminator.estimate(state0.center, state1.center)
sc = StateConfig.linear_2state(ld, invert=False)

In [ ]:
ax = plot_iq_blobs(iq, labels)

sc.plot(ax)

## Example 2: Latching 2-state discrimination

In [ ]:
from qiclib.hardware.recording import StateConfig

ld1 = LinearDiscriminator.through_points((0.2, 0.6), (0.2, -0.6))
ld2 = LinearDiscriminator.through_points((0.0, -0.6), (0.0, 0.6))

config = StateConfig.latching_2state(ld1, ld2)

In [ ]:
ax = plot_iq_blobs(iq, labels)

config.plot(ax, state=0)

## Example 3: 3 states discrimination

In [ ]:
from qiclib.state_estimation import LinearDiscriminator

state0 = BlobConfig(center=0.5 + 0.1j, chance=0.7, sigma=0.1)
state1 = BlobConfig(center=-0.3 + 0.1j, chance=0.2, sigma=0.1)
state2 = BlobConfig(center=-0.3 - 0.2j, chance=0.1, sigma=0.1)

iq, labels = synthetic_iq_blobs(state0, state1, state2, return_labels=True)

In [ ]:
ld1 = LinearDiscriminator.estimate(state0.center, state1.center)
ld2 = LinearDiscriminator.estimate(state0.center, state2.center)
ld3 = LinearDiscriminator.estimate(state1.center, state2.center)

cfg = StateConfig.linear_3states([ld1, ld2, ld3])

In [ ]:
ax = plot_iq_blobs(iq, labels)

cfg.plot(ax)